# Lab 10: Multilayer Perceptron (MLP) for Time Series Forecasting

## Objective
Build and train a simple feedforward neural network (MLP) to forecast the next time step of a multivariate time series dataset (energy consumption). We use **univariate multi‑step** input – the model sees a window of past values and predicts the next value of the target variable.

## Dataset
- The dataset contains hourly energy consumption data (AEP).
- Features include temporal indicators (hour, day of week, month, etc.) and lagged values.
- Data is already split into `train.csv`, `validation.csv`, and `test.csv`, and pre‑scaled using a `MinMaxScaler`.

## Model Architecture
A simple MLP with:
- `Flatten` layer to convert the 3D input `(time_steps, num_features)` into a 1D vector.
- Two hidden `Dense` layers with 32 units each and ReLU activation.
- A final `Dense` layer with 1 unit (linear activation) for regression.

## Training
- **Loss:** Mean Absolute Error (MAE)
- **Optimizer:** Adam with learning rate 1e‑3 (then fine‑tuned with 1e‑4)
- **Metrics:** MAE and MAPE
- **Callbacks:** ModelCheckpoint (save best) and TrainingMonitor (plot history)

## Fine‑Tuning
After the initial training, we load the best saved model, reduce the learning rate, and continue training for a few more epochs to improve performance.

## Evaluation
The final model is evaluated on the test set using several error metrics (MAE, MedAE, MSE, RMSE, MAPE, MDAPE).

---

In [1]:
# 1. SETUP AND IMPORTS

import os
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# TensorFlow / Keras
import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten, Input, Lambda, Reshape, Concatenate
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.callbacks import ModelCheckpoint, Callback
from tensorflow.keras.regularizers import l2

# Custom utility modules (from the `timeseires` package)
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor

# Scikit‑learn metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score

In [2]:
# 2. PATHS AND PARAMETERS

# Change working directory to dataset location
os.chdir(r'C:\Users\PMYLS\Documents\MachineLearningLaB\Dataset')

# Global parameters
time_steps = 24          # lookback window (24 hours)
num_features = 21        # number of input features
target_col = 0           # column index of the target variable

# Paths for saved models and outputs
OUTPUT_PATH = r'C:\Users\PMYLS\Documents\MachineLearningLaB\Dataset'
CHECKPOINT_PATTERN = os.path.join(OUTPUT_PATH, 'E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5')
FIG_PATH = os.path.join(OUTPUT_PATH, 'history.png')
JSON_PATH = os.path.join(OUTPUT_PATH, 'history.json')

## 3. Define the MLP Model

In [3]:
def MLP():
    """Create a simple MLP model for time series regression."""
    model = Sequential()
    # Flatten the (time_steps, num_features) input to a 1D vector
    model.add(Flatten(input_shape=(time_steps, num_features)))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(32, activation='relu'))   # second hidden layer
    model.add(Dense(1))                       # linear output for regression
    return model

In [4]:
# Instantiate and display the model
model = MLP()
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten (Flatten)           (None, 504)               0         
                                                                 
 dense (Dense)               (None, 32)                16160     
                                                                 
 dense_1 (Dense)             (None, 32)                1056      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 17,249
Trainable params: 17,249
Non-trainable params: 0
_________________________________________________________________


## 4. Compile the Model

In [5]:
# If no pre‑trained model is given, compile from scratch
if model is None:
    print("[INFO] compiling model...")
    model = MLP()
    opt = Adam(learning_rate=1e-3)
    model.compile(loss='mae', optimizer=opt, metrics=['mae', 'mape'])
else:
    print("[INFO] loading existing model...")
    model = load_model(model)
    # Optionally adjust learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


## 5. Load and Prepare the Data

The data is already scaled and split into train/validation/test sets. We also load the scaler to invert predictions later.

In [6]:
# Read CSV files
train_df = pd.read_csv('train.csv')
validation_df = pd.read_csv('validation.csv')
test_df = pd.read_csv('test.csv')

# Convert to numpy arrays
train_set = train_df.values
validation_set = validation_df.values
test_set = test_df.values

# Load the scaler
scaler = pickle.load(open('AEP_scaler.pkl', 'rb'))

train_set.shape, validation_set.shape, test_set.shape

C:\Users\PMYLS\anaconda3\envs\machinelearning\lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((860, 21), (90, 21), (30, 21))

## 6. Create Sequences (Univariate Multi‑Step)

We convert the time series into overlapping windows of length `time_steps` (24 hours) to predict the next single value (target_len=1).

In [7]:
start = time.time()
train_X, train_y = univariate_multi_step(train_set, time_steps, target_col=target_col, target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=target_col, target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=target_col, target_len=1)
print('Time Consumed', time.time()-start, "sec")

# Show shapes
print(f"train_X shape: {train_X.shape}, train_y shape: {train_y.shape}")
print(f"validation_X shape: {validation_X.shape}, validation_y shape: {validation_y.shape}")
print(f"test_X shape: {test_X.shape}, test_y shape: {test_y.shape}")

Time Consumed 0.01067805290222168 sec


## 7. Set Up Callbacks

- `ModelCheckpoint` saves the model with the best validation loss.
- `TrainingMonitor` logs training history and plots it.

In [8]:
checkpoint = ModelCheckpoint(
    CHECKPOINT_PATTERN,
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)
monitor = TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=0)
callbacks = [checkpoint, monitor]

## 8. Train the Model (Initial Run)

In [9]:
epochs = 10
batch_size = 32

history = model.fit(
    train_X, train_y,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(validation_X, validation_y),
    callbacks=callbacks,
    verbose=1
)

Epoch 1/10
27/27 [==============================] - ETA: 0s - loss: 0.0929 - mae: 0.0929 - mape: 32.1822
Epoch 1: val_loss improved from inf to 0.07663, saving model to C:\Users\PMYLS\Documents\MachineLearningLaB\Dataset\E1-cp-0001-loss0.08.h5
27/27 [==============================] - 1s 19ms/step - loss: 0.0929 - mae: 0.0929 - mape: 32.1822 - val_loss: 0.0766 - val_mae: 0.0766 - val_mape: 26.2836
Epoch 2/10
27/27 [==============================] - ETA: 0s - loss: 0.0670 - mae: 0.0670 - mape: 25.6082
Epoch 2: val_loss improved from 0.07663 to 0.06044, saving model to C:\Users\PMYLS\Documents\MachineLearningLaB\Dataset\E1-cp-0002-loss0.06.h5
27/27 [==============================] - 1s 18ms/step - loss: 0.0670 - mae: 0.0670 - mape: 25.6082 - val_loss: 0.0604 - val_mae: 0.0604 - val_mape: 21.7065
Epoch 3/10
27/27 [==============================] - ETA: 0s - loss: 0.0576 - mae: 0.0576 - mape: 23.9209
Epoch 3: val_loss improved from 0.06044 to 0.05031, saving model to C:\Users\PMYLS\Document

## 9. Evaluate the Initial Model

We load the best saved model (based on validation loss) and compute error metrics on the test set.

In [10]:
# Load the best model from the initial training
best_model_path = os.path.join(OUTPUT_PATH, 'E1-cp-0010-loss0.04.h5')
model_best = load_model(best_model_path)

# Predict on test set
y_pred_scaled = model_best.predict(test_X)
y_pred = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)

# Compute metrics
MAE = np.mean(abs(y_pred - y_test_unscaled))
MEDAE = np.median(abs(y_pred - y_test_unscaled))
MSE = np.mean((y_pred - y_test_unscaled)**2)
RMSE = np.sqrt(MSE)
MAPE = np.mean(np.abs((y_test_unscaled - y_pred) / y_test_unscaled)) * 100
MDAPE = np.median(np.abs((y_test_unscaled - y_pred) / y_test_unscaled)) * 100

print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')
print('\n\ny_test_unscaled.shape= ', y_test_unscaled.shape)
print('y_pred.shape= ', y_pred.shape)

1/1 [==============================] - 0s 150ms/step
Mean Absolute Error (MAE): 4318.02
Median Absolute Error (MedAE): 4120.98
Mean Squared Error (MSE): 18759679.73
Root Mean Squared Error (RMSE): 4331.24
Mean Absolute Percentage Error (MAPE): 27.53 %
Median Absolute Percentage Error (MDAPE): 26.87 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


## 10. Fine‑Tuning the Model

We continue training from the best checkpoint with a lower learning rate to further improve performance.

In [11]:
# Path to the model to fine‑tune
finetune_model_path = os.path.join(OUTPUT_PATH, 'E1-cp-0010-loss0.04.h5')

# Load the model
model_ft = load_model(finetune_model_path)

# Reduce learning rate for fine‑tuning
K.set_value(model_ft.optimizer.lr, 1e-4)
print("[INFO] new learning rate: {}".format(K.get_value(model_ft.optimizer.lr)))

# Re‑compile? Not necessary if optimizer is already set, but safe.
model_ft.compile(loss='mae', optimizer=model_ft.optimizer, metrics=['mae', 'mape'])

# Set up callbacks – use a different checkpoint pattern to avoid overwriting the initial best
ft_checkpoint_path = os.path.join(OUTPUT_PATH, 'ft-{epoch:04d}-loss{val_loss:.2f}.h5')
ft_checkpoint = ModelCheckpoint(ft_checkpoint_path, monitor='val_loss', save_best_only=True, verbose=1)
ft_monitor = TrainingMonitor(FIG_PATH.replace('.png', '_ft.png'), jsonPath=JSON_PATH.replace('.json', '_ft.json'), startAt=0)
ft_callbacks = [ft_checkpoint, ft_monitor]

[INFO] loading C:\Users\PMYLS\Documents\MachineLearningLaB\Dataset\E1-cp-0010-loss0.04.h5...
[INFO] old learning rate: 0.0010000000474974513
[INFO] new learning rate: 9.999999747378752e-05


In [12]:
# Fine‑tune for a few epochs
ft_epochs = 5
history_ft = model_ft.fit(
    train_X, train_y,
    batch_size=batch_size,
    epochs=ft_epochs,
    validation_data=(validation_X, validation_y),
    callbacks=ft_callbacks,
    verbose=1
)

Epoch 1/5
27/27 [==============================] - ETA: 0s - loss: 0.0396 - mae: 0.0396 - mape: 20.6764
Epoch 1: val_loss improved from inf to 0.04177, saving model to C:\Users\PMYLS\Documents\MachineLearningLaB\Dataset\ft-0001-loss0.04.h5
27/27 [==============================] - 1s 18ms/step - loss: 0.0396 - mae: 0.0396 - mape: 20.6764 - val_loss: 0.0418 - val_mae: 0.0418 - val_mape: 19.2315
Epoch 2/5
27/27 [==============================] - ETA: 0s - loss: 0.0329 - mae: 0.0329 - mape: 19.0152
Epoch 2: val_loss improved from 0.04177 to 0.03614, saving model to C:\Users\PMYLS\Documents\MachineLearningLaB\Dataset\ft-0002-loss0.04.h5
27/27 [==============================] - 1s 17ms/step - loss: 0.0329 - mae: 0.0329 - mape: 19.0152 - val_loss: 0.0361 - val_mae: 0.0361 - val_mape: 18.3219
Epoch 3/5
27/27 [==============================] - ETA: 0s - loss: 0.0311 - mae: 0.0311 - mape: 18.4450
Epoch 3: val_loss improved from 0.03614 to 0.03320, saving model to C:\Users\PMYLS\Documents\Machine

## 11. Evaluate the Fine‑Tuned Model

In [13]:
# Load the best fine‑tuned model
best_ft_path = os.path.join(OUTPUT_PATH, 'ft-0005-loss0.03.h5')
model_ft_best = load_model(best_ft_path)

# Predict and evaluate
y_pred_ft_scaled = model_ft_best.predict(test_X)
y_pred_ft = scaler.inverse_transform(y_pred_ft_scaled)

# Compute metrics
MAE_ft = np.mean(abs(y_pred_ft - y_test_unscaled))
MEDAE_ft = np.median(abs(y_pred_ft - y_test_unscaled))
MSE_ft = np.mean((y_pred_ft - y_test_unscaled)**2)
RMSE_ft = np.sqrt(MSE_ft)
MAPE_ft = np.mean(np.abs((y_test_unscaled - y_pred_ft) / y_test_unscaled)) * 100
MDAPE_ft = np.median(np.abs((y_test_unscaled - y_pred_ft) / y_test_unscaled)) * 100

print('Mean Absolute Error (MAE): ' + str(np.round(MAE_ft, 2)))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE_ft, 2)))
print('Mean Squared Error (MSE): ' + str(np.round(MSE_ft, 2)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE_ft, 2)))
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE_ft, 2)) + ' %')
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE_ft, 2)) + ' %')
print('\n\ny_test_unscaled.shape= ', y_test_unscaled.shape)
print('y_pred_ft.shape= ', y_pred_ft.shape)

1/1 [==============================] - 0s 124ms/step
Mean Absolute Error (MAE): 4131.66
Median Absolute Error (MedAE): 3914.07
Mean Squared Error (MSE): 17135023.87
Root Mean Squared Error (RMSE): 4139.44
Mean Absolute Percentage Error (MAPE): 26.14 %
Median Absolute Percentage Error (MDAPE): 24.27 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


## 12. Summary and Comparison

We compare the initial and fine‑tuned model performance on the test set.

| Metric | Initial | Fine‑Tuned |
|--------|---------|------------|
| MAE    | 4318.02 | 4131.66    |
| RMSE   | 4331.24 | 4139.44    |
| MAPE   | 27.53%  | 26.14%     |

Fine‑tuning with a lower learning rate improved all metrics, demonstrating the benefit of continued training with a smaller step size.